# Hukuk Asistanı — E5-large Embedding (Colab GPU)

Bu notebook `chunks.csv`'yi alır, **intfloat/multilingual-e5-large** ile GPU'da embed eder ve
`embeddings.npy`, `embedding_metadata.json`, `index.faiss`, `chunk_mapping.pkl` üretip zip'ler.

**Önce:** Colab menüsü → Runtime → Change runtime type → **T4 GPU** seçin.

Sonra hücreleri sırayla çalıştırın. Süre: T4'te ~15-40 dk (554K chunk).


In [ ]:
!pip -q install sentence-transformers faiss-cpu pandas numpy


## 1) chunks.csv yükleyin
Aşağıdaki hücre bir dosya seçtirir — yerelde `data/processed/chunks.csv`'yi seçin.


In [ ]:
from google.colab import files
up = files.upload()   # chunks.csv seçin
print('yüklendi:', list(up.keys()))


## 2) Embed + index üret
Daha hızlı ama biraz daha düşük kalite isterseniz `MODEL` satırını
`sentence-transformers/paraphrase-multilingual-mpnet-base-v2` yapın (prefix'i boşaltın).


In [ ]:
import pandas as pd, numpy as np, pickle, json, datetime, torch, faiss
from sentence_transformers import SentenceTransformer

MODEL = 'intfloat/multilingual-e5-large'   # 1024-dim
PASSAGE_PREFIX = 'passage: '               # E5 gerektirir; mpnet için '' yapın
BATCH = 32                                 # T4 OOM verirse düşürün

df = pd.read_csv('chunks.csv')
df['text'] = df['text'].fillna('').astype(str)
texts = [PASSAGE_PREFIX + t for t in df['text'].tolist()]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| chunks:', len(texts))
assert dev == 'cuda', 'GPU açık değil! Runtime > Change runtime type > T4 GPU'

model = SentenceTransformer(MODEL, device=dev)
emb = model.encode(texts, batch_size=BATCH, show_progress_bar=True,
                   convert_to_numpy=True, normalize_embeddings=True).astype('float32')
print('embeddings:', emb.shape)

np.save('embeddings.npy', emb)
json.dump({'total_chunks': int(emb.shape[0]), 'dimension': int(emb.shape[1]),
           'model': MODEL, 'normalized': True,
           'date': datetime.date.today().isoformat()},
          open('embedding_metadata.json','w'), ensure_ascii=False, indent=2)

index = faiss.IndexFlatL2(emb.shape[1]); index.add(emb)
faiss.write_index(index, 'index.faiss')

mapping = {int(r.chunk_id): {'text': str(r.text), 'section': str(r.section),
           'original_doc_id': str(r.original_doc_id), 'is_atomic': bool(r.is_atomic)}
           for r in df.itertuples()}
pickle.dump(mapping, open('chunk_mapping.pkl','wb'))
print('OK — index:', index.ntotal, 'vectors, dim', emb.shape[1])


## 3) İndir
Zip'i indirip yerelde `data/processed/` içine açın (mevcut dosyaların üzerine yazın).


In [ ]:
!zip -q e5_artifacts.zip embeddings.npy embedding_metadata.json index.faiss chunk_mapping.pkl
from google.colab import files
files.download('e5_artifacts.zip')


## 4) Yerelde
```bash
# e5_artifacts.zip içindeki 4 dosyayı data/processed/ içine çıkarın, sonra:
set ST_MODEL_NAME=intfloat/multilingual-e5-large   # Windows (kalıcı için .env'e yazın)
python src/07_search_engine.py "kefilin sorumluluğunun kapsamı ve süresi"
python src/try_queries.py --n 3 --snippet 200
python src/09_benchmark_test.py
```
`config.py` E5'i model adından otomatik algılar (dim=1024, query:/passage: prefix).
Not: E5 kullanırken sorgular otomatik `query: ` prefix'i alır — bu yüzden `ST_MODEL_NAME`'i
E5 yaptığınızdan emin olun, yoksa index (1024) ile model (384) boyutu uyuşmaz.
